State is continuous (position, velocity)
Noise is Gaussian
System is linear

In [1]:
import numpy as np

# Initial state [position, velocity]
x = np.array([[0],
              [1]])

# Initial covariance
P = np.eye(2)

# State transition matrix
A = np.array([[1, 1],
              [0, 1]])

# Control matrix
B = np.array([[0],
              [0]])

# Measurement matrix
H = np.array([[1, 0]])

# Process noise
Q = np.eye(2) * 0.01

# Measurement noise
R = np.array([[0.1]])

measurements = [1, 2, 3, 4, 5]

for z in measurements:
    # Predict
    x = A @ x
    P = A @ P @ A.T + Q

    # Update
    K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
    x = x + K @ (z - H @ x)
    P = (np.eye(2) - K @ H) @ P

    print("State:\n", x)

State:
 [[1.]
 [1.]]
State:
 [[2.]
 [1.]]
State:
 [[3.]
 [1.]]
State:
 [[4.]
 [1.]]
State:
 [[5.]
 [1.]]


In [2]:
import numpy as np

def f(x):
    # Nonlinear motion model
    return np.array([
        x[0] + x[1],
        x[1]
    ])

def h(x):
    # Nonlinear measurement model
    return np.array([np.sqrt(x[0]**2)])

x = np.array([0.0, 1.0])
P = np.eye(2)
Q = np.eye(2) * 0.01
R = np.array([[0.1]])

for z in [1,2,3,4]:
    # Jacobian of f
    F = np.array([[1,1],
                  [0,1]])

    # Predict
    x = f(x)
    P = F @ P @ F.T + Q

    # Jacobian of h
    H = np.array([[x[0]/np.sqrt(x[0]**2), 0]])

    # Update
    K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
    x = x + K @ (z - h(x))
    P = (np.eye(2) - K @ H) @ P

    print("EKF State:", x)

EKF State: [1. 1.]
EKF State: [2. 1.]
EKF State: [3. 1.]
EKF State: [4. 1.]


In [3]:
import numpy as np

def f(x):
    return np.array([x[0] + x[1], x[1]])

def h(x):
    return np.array([x[0]])

x = np.array([0., 1.])
P = np.eye(2)
Q = np.eye(2) * 0.01
R = np.array([[0.1]])

for z in [1,2,3,4]:
    # Sigma points
    sigma_points = [
        x,
        x + np.sqrt(P.diagonal()),
        x - np.sqrt(P.diagonal())
    ]

    # Predict
    sigma_pred = [f(s) for s in sigma_points]
    x_pred = np.mean(sigma_pred, axis=0)

    # Covariance
    P = np.cov(np.array(sigma_pred).T) + Q

    # Update
    z_pred = h(x_pred)
    K = P @ np.array([[1],[0]]) @ np.linalg.inv(R + P[0,0])
    x = x_pred + K.flatten() * (z - z_pred)
    
    print("UKF State:", x)

UKF State: [1. 1.]
UKF State: [2. 1.]
UKF State: [3. 1.]
UKF State: [4. 1.]


In [4]:
import numpy as np

# initial state
x = np.array([[0],
              [1]])

# initial covariance
P = np.eye(2)

# Convert to Information form
Omega = np.linalg.inv(P)
xi = Omega @ x

# Measurement matrix
H = np.array([[1, 0]])

# Measurement noise
R = np.array([[0.1]])

measurements = [1, 2, 3, 4]

for z in measurements:
    # Information update
    Omega = Omega + H.T @ np.linalg.inv(R) @ H
    xi = xi + H.T @ np.linalg.inv(R) @ np.array([[z]])

    # Recover state
    x = np.linalg.inv(Omega) @ xi

    print("State:", x)

State: [[0.90909091]
 [1.        ]]
State: [[1.42857143]
 [1.        ]]
State: [[1.93548387]
 [1.        ]]
State: [[2.43902439]
 [1.        ]]


In [5]:
import numpy as np

# nonlinear motion model
def f(x):
    return np.array([
        x[0] + x[1],
        x[1]
    ])

# nonlinear measurement model
def h(x):
    return np.array([np.sqrt(x[0]**2)])

# initial state
x = np.array([0.1, 1.0])
P = np.eye(2)

# convert to information form
Omega = np.linalg.inv(P)
xi = Omega @ x

Q = np.eye(2) * 0.01
R = np.array([[0.1]])

measurements = [1, 2, 3, 4]

for z in measurements:
    # Prediction
    x = f(x)

    # Jacobian of measurement
    H = np.array([[x[0]/np.sqrt(x[0]**2), 0]])

    # Information update
    Omega = Omega + H.T @ np.linalg.inv(R) @ H
    xi = xi + H.T @ np.linalg.inv(R) @ (np.array([z]) - h(x) + H @ x)

    # Recover state
    x = np.linalg.inv(Omega) @ xi

    print("EIF State:", x)

EIF State: [0.91818182 1.        ]
EIF State: [1.43333333 1.        ]
EIF State: [1.93870968 1.        ]
EIF State: [2.44146341 1.        ]
